# 🏦 JPMC Consumer Banking - Hands-on Lab
## Notebook 02: Data Ingestion

---

### What You'll Learn
- **Schema Detection** using `INFER_SCHEMA` to auto-discover Parquet columns
- **COPY INTO** (batch ingestion) with `MATCH_BY_COLUMN_NAME`
- **Schema Evolution** - enabling tables to auto-expand when new columns arrive
- **Native Tables** vs **Managed Iceberg Tables** - when to use which
- **Snowpipe** (micro-batch) - auto-ingest with notifications
- **Snowpipe Streaming** - real-time channel-based ingestion

### Ingestion Methods Comparison

| Method | Latency | Best For | Trigger |
|--------|---------|----------|---------|
| COPY INTO | Minutes | Bulk/batch loads | Manual or Task-scheduled |
| Snowpipe | Seconds-Minutes | Continuous file landing | Cloud event notification |
| Snowpipe Streaming | Sub-second | Real-time event streams | SDK push (row-level) |
| Connectors (Kafka) | Seconds | Streaming platforms | Kafka topic consumption |

In [ ]:
-- Setup context
USE ROLE ACCOUNTADMIN;
USE DATABASE JPMC_CONSUMER_BANKING_HOL;
USE SCHEMA HOL_LAB;
USE WAREHOUSE HOL_MEDIUM_WH;

## Step 1: Schema Detection with INFER_SCHEMA

Before loading data, we can auto-discover the schema from Parquet files. This eliminates manual DDL writing.

In [ ]:
-- =============================================================================
-- SCHEMA DETECTION - Automatically discover schema from Parquet files
-- =============================================================================

-- Infer schema for CUSTOMERS
SELECT *
FROM TABLE(
    INFER_SCHEMA(
        LOCATION => '@HOL_PARQUET_STAGE/customers/',
        FILE_FORMAT => 'HOL_PARQUET_FF'
    )
);

In [ ]:
-- Generate CREATE TABLE DDL from inferred schema (useful for documentation)
SELECT GENERATE_COLUMN_DESCRIPTION(ARRAY_AGG(OBJECT_CONSTRUCT(*)), 'TABLE') AS DDL_COLUMNS
FROM TABLE(
    INFER_SCHEMA(
        LOCATION => '@HOL_PARQUET_STAGE/customers/',
        FILE_FORMAT => 'HOL_PARQUET_FF'
    )
);

## Step 2: Create Native Tables & Load with COPY INTO

These tables use Snowflake's native storage format. Best for frequently queried, high-performance workloads.

**Tables created as NATIVE:** CUSTOMERS, TRANSACTIONS, SUPPORT_TICKETS

In [ ]:
-- =============================================================================
-- NATIVE TABLE: CUSTOMERS (using CREATE TABLE with columns from inferred schema)
-- =============================================================================

-- Create table using INFER_SCHEMA output (schema detection + table creation)
CREATE OR REPLACE TABLE CUSTOMERS
    USING TEMPLATE (
        SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
        FROM TABLE(
            INFER_SCHEMA(
                LOCATION => '@HOL_PARQUET_STAGE/customers/',
                FILE_FORMAT => 'HOL_PARQUET_FF'
            )
        )
    );

-- Enable schema evolution (auto-add new columns if future files have them)
ALTER TABLE CUSTOMERS SET ENABLE_SCHEMA_EVOLUTION = TRUE;

-- Load data using COPY INTO with column name matching
COPY INTO CUSTOMERS
FROM @HOL_PARQUET_STAGE/customers/
FILE_FORMAT = 'HOL_PARQUET_FF'
MATCH_BY_COLUMN_NAME = 'CASE_INSENSITIVE';

SELECT COUNT(*) as row_count, 'CUSTOMERS' as table_name FROM CUSTOMERS;

In [ ]:
-- =============================================================================
-- NATIVE TABLE: TRANSACTIONS
-- =============================================================================

CREATE OR REPLACE TABLE TRANSACTIONS
    USING TEMPLATE (
        SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
        FROM TABLE(
            INFER_SCHEMA(
                LOCATION => '@HOL_PARQUET_STAGE/transactions/',
                FILE_FORMAT => 'HOL_PARQUET_FF'
            )
        )
    );

ALTER TABLE TRANSACTIONS SET ENABLE_SCHEMA_EVOLUTION = TRUE;

COPY INTO TRANSACTIONS
FROM @HOL_PARQUET_STAGE/transactions/
FILE_FORMAT = 'HOL_PARQUET_FF'
MATCH_BY_COLUMN_NAME = 'CASE_INSENSITIVE';

SELECT COUNT(*) as row_count, 'TRANSACTIONS' as table_name FROM TRANSACTIONS;

In [ ]:
-- =============================================================================
-- NATIVE TABLE: SUPPORT_TICKETS
-- =============================================================================

CREATE OR REPLACE TABLE SUPPORT_TICKETS
    USING TEMPLATE (
        SELECT ARRAY_AGG(OBJECT_CONSTRUCT(*))
        FROM TABLE(
            INFER_SCHEMA(
                LOCATION => '@HOL_PARQUET_STAGE/support_tickets/',
                FILE_FORMAT => 'HOL_PARQUET_FF'
            )
        )
    );

ALTER TABLE SUPPORT_TICKETS SET ENABLE_SCHEMA_EVOLUTION = TRUE;

COPY INTO SUPPORT_TICKETS
FROM @HOL_PARQUET_STAGE/support_tickets/
FILE_FORMAT = 'HOL_PARQUET_FF'
MATCH_BY_COLUMN_NAME = 'CASE_INSENSITIVE';

SELECT COUNT(*) as row_count, 'SUPPORT_TICKETS' as table_name FROM SUPPORT_TICKETS;

## Step 3: Create Managed Iceberg Tables & Load

Iceberg tables store data in Apache Iceberg open table format. Benefits:
- **Open format** - readable by Spark, Trino, Flink without Snowflake compute
- **Interoperability** - query from any engine via Horizon REST Catalog
- **Time travel** - built-in snapshot management

**Tables created as ICEBERG:** ACCOUNTS, LOANS, CREDIT_CARDS

In [ ]:
-- =============================================================================
-- MANAGED ICEBERG TABLE: ACCOUNTS
-- Snowflake manages the catalog, storage, and compaction
-- =============================================================================

CREATE OR REPLACE ICEBERG TABLE ACCOUNTS (
    ACCOUNT_ID VARCHAR,
    CUSTOMER_ID VARCHAR,
    ACCOUNT_TYPE VARCHAR,
    BALANCE FLOAT,
    AVAILABLE_BALANCE FLOAT,
    INTEREST_RATE FLOAT,
    OPEN_DATE VARCHAR,
    STATUS VARCHAR,
    BRANCH_ID VARCHAR,
    OVERDRAFT_PROTECTION BOOLEAN,
    LAST_ACTIVITY_DATE VARCHAR
)
    CATALOG = 'SNOWFLAKE'
    EXTERNAL_VOLUME = 'ICEBERG_EXTERNAL_VOLUME'
    BASE_LOCATION = 'jpmc_hol/accounts/';

-- Load data into Iceberg table
COPY INTO ACCOUNTS
FROM @HOL_PARQUET_STAGE/accounts/
FILE_FORMAT = 'HOL_PARQUET_FF'
MATCH_BY_COLUMN_NAME = 'CASE_INSENSITIVE';

SELECT COUNT(*) as row_count, 'ACCOUNTS (Iceberg)' as table_name FROM ACCOUNTS;

In [ ]:
-- =============================================================================
-- MANAGED ICEBERG TABLE: LOANS
-- =============================================================================

CREATE OR REPLACE ICEBERG TABLE LOANS (
    LOAN_ID VARCHAR,
    CUSTOMER_ID VARCHAR,
    LOAN_TYPE VARCHAR,
    PRINCIPAL_AMOUNT FLOAT,
    CURRENT_BALANCE FLOAT,
    INTEREST_RATE FLOAT,
    TERM_MONTHS INT,
    MONTHLY_PAYMENT FLOAT,
    ORIGINATION_DATE VARCHAR,
    MATURITY_DATE VARCHAR,
    STATUS VARCHAR,
    CREDIT_SCORE_AT_ORIGINATION INT,
    DEBT_TO_INCOME_RATIO FLOAT,
    COLLATERAL_VALUE FLOAT
)
    CATALOG = 'SNOWFLAKE'
    EXTERNAL_VOLUME = 'ICEBERG_EXTERNAL_VOLUME'
    BASE_LOCATION = 'jpmc_hol/loans/';

COPY INTO LOANS
FROM @HOL_PARQUET_STAGE/loans/
FILE_FORMAT = 'HOL_PARQUET_FF'
MATCH_BY_COLUMN_NAME = 'CASE_INSENSITIVE';

SELECT COUNT(*) as row_count, 'LOANS (Iceberg)' as table_name FROM LOANS;

In [ ]:
-- =============================================================================
-- MANAGED ICEBERG TABLE: CREDIT_CARDS
-- =============================================================================

CREATE OR REPLACE ICEBERG TABLE CREDIT_CARDS (
    CARD_ID VARCHAR,
    CUSTOMER_ID VARCHAR,
    CARD_TYPE VARCHAR,
    CARD_NUMBER_HASH VARCHAR,
    CREDIT_LIMIT INT,
    CURRENT_BALANCE FLOAT,
    AVAILABLE_CREDIT FLOAT,
    APR FLOAT,
    MINIMUM_PAYMENT FLOAT,
    PAYMENT_DUE_DATE VARCHAR,
    REWARDS_POINTS INT,
    CASH_BACK_EARNED FLOAT,
    ISSUE_DATE VARCHAR,
    EXPIRATION_DATE VARCHAR,
    STATUS VARCHAR,
    AUTOPAY_ENABLED BOOLEAN
)
    CATALOG = 'SNOWFLAKE'
    EXTERNAL_VOLUME = 'ICEBERG_EXTERNAL_VOLUME'
    BASE_LOCATION = 'jpmc_hol/credit_cards/';

COPY INTO CREDIT_CARDS
FROM @HOL_PARQUET_STAGE/credit_cards/
FILE_FORMAT = 'HOL_PARQUET_FF'
MATCH_BY_COLUMN_NAME = 'CASE_INSENSITIVE';

SELECT COUNT(*) as row_count, 'CREDIT_CARDS (Iceberg)' as table_name FROM CREDIT_CARDS;

In [ ]:
-- =============================================================================
-- VIEW ICEBERG FILES ON EXTERNAL STORAGE (S3)
-- These are the actual Parquet data files and metadata managed by Iceberg
-- =============================================================================

-- ACCOUNTS Iceberg files
SELECT 'ACCOUNTS' AS TABLE_NAME, * FROM TABLE(
    INFORMATION_SCHEMA.ICEBERG_TABLE_FILES(
        TABLE_NAME => 'ACCOUNTS'
    )
);

-- LOANS Iceberg files
SELECT 'LOANS' AS TABLE_NAME, * FROM TABLE(
    INFORMATION_SCHEMA.ICEBERG_TABLE_FILES(
        TABLE_NAME => 'LOANS'
    )
);

-- CREDIT_CARDS Iceberg files
SELECT 'CREDIT_CARDS' AS TABLE_NAME, * FROM TABLE(
    INFORMATION_SCHEMA.ICEBERG_TABLE_FILES(
        TABLE_NAME => 'CREDIT_CARDS'
    )
);

## Step 4: Schema Evolution Demo

Schema Evolution allows tables to automatically adapt when new columns appear in incoming data. Watch how a new column gets added automatically.

In [ ]:
-- =============================================================================
-- SCHEMA EVOLUTION DEMO
-- Show how tables automatically adapt to new columns in incoming data
-- =============================================================================

-- Check current schema of CUSTOMERS table
DESCRIBE TABLE CUSTOMERS;

In [ ]:
# Simulate schema evolution: Create a new Parquet file with an additional column
import pandas as pd
from snowflake.snowpark.context import get_active_session

session = get_active_session()

# Create a small DataFrame with a NEW column 'LOYALTY_TIER' that doesn't exist yet
evolution_data = pd.DataFrame({
    'CUSTOMER_ID': ['CUST00000001', 'CUST00000002', 'CUST00000003'],
    'FIRST_NAME': ['James', 'Mary', 'Robert'],
    'LAST_NAME': ['Smith', 'Johnson', 'Williams'],
    'EMAIL': ['james.smith1@email.com', 'mary.johnson2@email.com', 'robert.williams3@email.com'],
    'PHONE': ['+12125551234', '+13105555678', '+17135559012'],
    'SSN_HASH': ['abc123', 'def456', 'ghi789'],
    'DATE_OF_BIRTH': ['1980-05-15', '1975-11-22', '1990-03-08'],
    'ADDRESS_JSON': ['{"street":"123 Main St","city":"New York","state":"NY","zip":"10001"}'] * 3,
    'SEGMENT': ['Premium', 'Gold', 'Standard'],
    'RISK_RATING': ['Low', 'Medium', 'Low'],
    'CREATED_AT': ['2020-01-15 10:30:00', '2019-06-22 14:00:00', '2021-03-01 09:00:00'],
    'IS_ACTIVE': [True, True, False],
    # NEW COLUMNS - these will trigger schema evolution
    'LOYALTY_TIER': ['DIAMOND', 'PLATINUM', 'GOLD'],
    'PREFERRED_LANGUAGE': ['EN', 'ES', 'EN']
})

# Write to stage as new Parquet file
snow_df = session.create_dataframe(evolution_data)
snow_df.write.mode("overwrite").save_as_table("HOL_STAGING.STG_CUSTOMERS_EVOLVED")

session.sql("""
    COPY INTO @HOL_PARQUET_STAGE/customers_evolved/
    FROM HOL_STAGING.STG_CUSTOMERS_EVOLVED
    FILE_FORMAT = (TYPE = 'PARQUET')
    OVERWRITE = TRUE
    HEADER = TRUE
""").collect()

print("✅ Created Parquet file with 2 new columns: LOYALTY_TIER, PREFERRED_LANGUAGE")

In [ ]:
-- Now load the evolved file - the table will AUTO-ADD the new columns!
COPY INTO CUSTOMERS
FROM @HOL_PARQUET_STAGE/customers_evolved/
FILE_FORMAT = 'HOL_PARQUET_FF'
MATCH_BY_COLUMN_NAME = 'CASE_INSENSITIVE';

-- Verify: LOYALTY_TIER and PREFERRED_LANGUAGE columns were added automatically
DESCRIBE TABLE CUSTOMERS;

In [ ]:
-- Check the new columns - existing rows have NULL, new rows have values
SELECT CUSTOMER_ID, FIRST_NAME, LOYALTY_TIER, PREFERRED_LANGUAGE
FROM CUSTOMERS
WHERE LOYALTY_TIER IS NOT NULL
LIMIT 5;

## Step 5: Snowpipe (Micro-Batch Ingestion)

Snowpipe provides continuous, serverless data loading triggered by cloud storage event notifications. Files are loaded within seconds-minutes of landing.

**How it works:**
1. Files land in a stage (S3 bucket, Azure Blob, GCS)
2. Cloud event notification triggers Snowpipe
3. Snowpipe loads files automatically using a serverless compute model
4. No warehouse needed - billed per-file (serverless credits)

In [ ]:
-- =============================================================================
-- SNOWPIPE SETUP (Micro-batch ingestion)
-- =============================================================================

-- Step 1: Create a notification integration (AWS SQS for auto-ingest)
-- NOTE: In production, replace with your actual SQS ARN
-- CREATE OR REPLACE NOTIFICATION INTEGRATION HOL_SNOWPIPE_NOTIFICATION
--     ENABLED = TRUE
--     TYPE = QUEUE
--     NOTIFICATION_PROVIDER = AWS_SQS
--     DIRECTION = INBOUND
--     AWS_SQS_ARN = 'arn:aws:sqs:us-east-1:123456789012:hol-snowpipe-queue'
--     AWS_SQS_ROLE_ARN = 'arn:aws:iam::123456789012:role/snowpipe-role';

-- Step 2: Create a target table for Snowpipe
CREATE OR REPLACE TABLE TRANSACTIONS_STREAMING_PIPE (
    TXN_ID VARCHAR,
    ACCOUNT_ID VARCHAR,
    TXN_DATE TIMESTAMP,
    TXN_TYPE VARCHAR,
    AMOUNT FLOAT,
    CURRENCY VARCHAR,
    MERCHANT VARCHAR,
    CATEGORY VARCHAR,
    CHANNEL VARCHAR,
    STATUS VARCHAR,
    REFERENCE_NUM VARCHAR,
    DESCRIPTION VARCHAR,
    _LOADED_AT TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

-- Step 3: Create the Snowpipe (points to a dedicated subfolder for new files)
CREATE OR REPLACE PIPE HOL_TRANSACTIONS_PIPE
    AUTO_INGEST = TRUE
    -- INTEGRATION = 'HOL_SNOWPIPE_NOTIFICATION'  -- uncomment with real AWS SQS integration
    COMMENT = 'Auto-ingest pipe for transaction files landing in stage'
AS
    COPY INTO TRANSACTIONS_STREAMING_PIPE
    FROM @HOL_PARQUET_STAGE/transactions_snowpipe/
    FILE_FORMAT = 'HOL_PARQUET_FF'
    MATCH_BY_COLUMN_NAME = 'CASE_INSENSITIVE';

-- View pipe status
SHOW PIPES LIKE 'HOL_%';

In [ ]:
-- First, place a NEW file in the stage so Snowpipe has something to load
-- (Snowpipe only loads files it hasn't seen before)
COPY INTO @HOL_PARQUET_STAGE/transactions_snowpipe/
FROM (
    SELECT * FROM TRANSACTIONS LIMIT 1000
)
FILE_FORMAT = (TYPE = 'PARQUET')
OVERWRITE = TRUE
HEADER = TRUE;

In [ ]:
-- Now trigger Snowpipe to pick up the new file
ALTER PIPE HOL_TRANSACTIONS_PIPE REFRESH;

-- Check pipe status (pendingFileCount should show files queued)
SELECT SYSTEM$PIPE_STATUS('HOL_TRANSACTIONS_PIPE');

In [ ]:
-- Wait ~30 seconds for Snowpipe to load, then check results
-- View copy history for the pipe
SELECT *
FROM TABLE(INFORMATION_SCHEMA.COPY_HISTORY(
    TABLE_NAME => 'TRANSACTIONS_STREAMING_PIPE',
    START_TIME => DATEADD(HOUR, -1, CURRENT_TIMESTAMP())
))
ORDER BY LAST_LOAD_TIME DESC
LIMIT 10;

In [ ]:
-- Verify rows were loaded by Snowpipe
SELECT COUNT(*) AS rows_loaded_by_snowpipe FROM TRANSACTIONS_STREAMING_PIPE;

## Step 8: Verify All Loaded Tables

In [ ]:
-- =============================================================================
-- VERIFICATION: All tables loaded successfully
-- =============================================================================

SELECT 'CUSTOMERS (Native)' as TABLE_NAME, COUNT(*) as ROW_COUNT FROM CUSTOMERS
UNION ALL
SELECT 'TRANSACTIONS (Native)', COUNT(*) FROM TRANSACTIONS
UNION ALL
SELECT 'SUPPORT_TICKETS (Native)', COUNT(*) FROM SUPPORT_TICKETS
UNION ALL
SELECT 'ACCOUNTS (Iceberg)', COUNT(*) FROM ACCOUNTS
UNION ALL
SELECT 'LOANS (Iceberg)', COUNT(*) FROM LOANS
UNION ALL
SELECT 'CREDIT_CARDS (Iceberg)', COUNT(*) FROM CREDIT_CARDS
ORDER BY TABLE_NAME;

In [ ]:
-- Verify Iceberg table metadata
SELECT SYSTEM$GET_ICEBERG_TABLE_INFORMATION('ACCOUNTS') as ICEBERG_METADATA;

## Summary

### Ingestion Methods Demonstrated

| Method | Tables Loaded | Key Feature |
|--------|--------------|-------------|
| **COPY INTO** (batch) | CUSTOMERS, TRANSACTIONS, SUPPORT_TICKETS, ACCOUNTS, LOANS, CREDIT_CARDS | Schema detection + column matching |
| **Schema Evolution** | CUSTOMERS (added LOYALTY_TIER, PREFERRED_LANGUAGE) | Auto-expand on new columns |
| **Snowpipe** | TRANSACTIONS_STREAMING_PIPE | Auto-ingest with cloud notifications |

### Table Types Created

| Table | Storage Type | Reason |
|-------|-------------|--------|
| CUSTOMERS | Native | High-frequency queries, schema evolution demo |
| TRANSACTIONS | Native | Highest volume, analytics workload |
| SUPPORT_TICKETS | Native | Unstructured text for Cortex AI processing |
| ACCOUNTS | Managed Iceberg | Interoperability demo (Spark access) |
| LOANS | Managed Iceberg | Interoperability demo (Spark access) |
| CREDIT_CARDS | Managed Iceberg | Interoperability demo (Spark access) |

---

**Next:** Proceed to `03_Transformation_Dynamic_Tables.ipynb` to build a transformation DAG using Dynamic Tables.